# 02 · Build and verify AnnData

Attach the Stage 01 output as a Kaggle notebook input. Assemble a cell-by-gene matrix with raw counts, Ensembl identifiers and source donor, condition and annotation fields. The validated H5AD and its manifest feed Stage 03.


## Locate the Stage 01 checkpoint


In [1]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

print("Kaggle input exists:", INPUT_DIR.is_dir())

print("\nAttached inputs:")
for item in sorted(INPUT_DIR.iterdir()):
    print(" -", item.name)

# Tìm bảng ánh xạ cell đã được xác minh ở Stage 01
mapping_files = sorted(
    INPUT_DIR.rglob("cell_mapping.tsv.gz")
)

print("\nCell mapping files found:", len(mapping_files))

for path in mapping_files:
    print(path)

# Xác định các checkpoint có cấu trúc phù hợp
required_raw = [
    "GSE96583_batch2.total.tsne.df.tsv.gz",
    "GSE96583_batch2.genes.tsv.gz",
    "GSM2560248_barcodes.tsv.gz",
    "GSM2560249_barcodes.tsv.gz",
    "GSM2560248_2.1.mtx.gz",
    "GSM2560249_2.2.mtx.gz",
]

required_audit = [
    "cell_mapping.tsv.gz",
    "raw_count_checks.tsv",
    "source_reference.json",
]

for mapping_path in mapping_files:
    source_project = mapping_path.parents[2]

    print("\nCandidate source project:", source_project)

    for filename in required_raw:
        path = source_project / "data" / "raw" / filename
        print("OK" if path.is_file() else "MISSING", filename)

    for filename in required_audit:
        path = source_project / "results" / "audit" / filename
        print("OK" if path.is_file() else "MISSING", filename)

Kaggle input exists: True

Attached inputs:
 - notebooks

Cell mapping files found: 1
/kaggle/input/notebooks/tthanhh811/01-ifnb-data-access/kang_ifnb/results/audit/cell_mapping.tsv.gz

Candidate source project: /kaggle/input/notebooks/tthanhh811/01-ifnb-data-access/kang_ifnb
OK GSE96583_batch2.total.tsne.df.tsv.gz
OK GSE96583_batch2.genes.tsv.gz
OK GSM2560248_barcodes.tsv.gz
OK GSM2560249_barcodes.tsv.gz
OK GSM2560248_2.1.mtx.gz
OK GSM2560249_2.2.mtx.gz
OK cell_mapping.tsv.gz
OK raw_count_checks.tsv
OK source_reference.json


## 1. Environment and project initialization

Locate archived inputs and record the starting Python environment.


In [2]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

# --------------------------------------------------
# 1. Input: Stage 01 checkpoint (read-only)
# --------------------------------------------------

assert len(mapping_files) == 1, (
    "Expected exactly one Stage 01 checkpoint."
)

SOURCE_PROJECT = mapping_files[0].parents[2]

RAW_DIR = SOURCE_PROJECT / "data" / "raw"
SOURCE_AUDIT_DIR = SOURCE_PROJECT / "results" / "audit"

# --------------------------------------------------
# 2. Output: Stage 02 (writable)
# --------------------------------------------------

PROJECT = Path("/kaggle/working/kang_ifnb")

PROCESSED_DIR = PROJECT / "data" / "processed"
AUDIT_DIR = PROJECT / "results" / "audit"

for folder in [PROCESSED_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 3. Verify directories
# --------------------------------------------------

assert RAW_DIR.is_dir()
assert SOURCE_AUDIT_DIR.is_dir()
assert PROCESSED_DIR.is_dir()
assert AUDIT_DIR.is_dir()

print("SOURCE:", SOURCE_PROJECT)
print("RAW:", RAW_DIR)
print("SOURCE AUDIT:", SOURCE_AUDIT_DIR)
print("PROCESSED:", PROCESSED_DIR)
print("STAGE 02 AUDIT:", AUDIT_DIR)

# --------------------------------------------------
# 4. Inspect actual Python environment
# --------------------------------------------------

packages = {}

for package in [
    "numpy", "pandas", "scipy", "h5py", "anndata"
]:
    try:
        packages[package] = version(package)
    except PackageNotFoundError:
        packages[package] = None

runtime = {
    "recorded_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version.split()[0],
    "packages": packages,
}

runtime_path = (
    AUDIT_DIR / "stage02_runtime_before_install.json"
)

runtime_path.write_text(
    json.dumps(runtime, indent=2) + "\n",
    encoding="utf-8",
)

print("\nRuntime:")
print(json.dumps(runtime, indent=2))

print("\nSTAGE 02 INITIALIZATION: PASS")

SOURCE: /kaggle/input/notebooks/tthanhh811/01-ifnb-data-access/kang_ifnb
RAW: /kaggle/input/notebooks/tthanhh811/01-ifnb-data-access/kang_ifnb/data/raw
SOURCE AUDIT: /kaggle/input/notebooks/tthanhh811/01-ifnb-data-access/kang_ifnb/results/audit
PROCESSED: /kaggle/working/kang_ifnb/data/processed
STAGE 02 AUDIT: /kaggle/working/kang_ifnb/results/audit

Runtime:
{
  "recorded_at_utc": "2026-11-07T15:42:39.537996+00:00",
  "python": "3.13.18",
  "packages": {
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "h5py": "3.14.0",
    "anndata": null
  }
}

STAGE 02 INITIALIZATION: PASS


## 2. AnnData installation and dependency verification

Install pinned AnnData and check that core package versions are preserved.


In [3]:
%pip install "anndata==0.13.4"

In [4]:
import json
from datetime import datetime, timezone
from importlib.metadata import version

import anndata as ad
import numpy as np
import pandas as pd
import scipy
import h5py

# Read Stage 02 environment before installation
before_path = (
    AUDIT_DIR / "stage02_runtime_before_install.json"
)

assert before_path.is_file()
before = json.loads(before_path.read_text())

# Collect installed versions
packages_after = {
    package: version(package)
    for package in [
        "anndata", "numpy", "pandas", "scipy", "h5py"
    ]
}

assert packages_after["anndata"] == "0.13.4"

# Check whether existing dependencies changed
changes = {}

for name in ["numpy", "pandas", "scipy", "h5py"]:
    old = before["packages"][name]
    new = packages_after[name]

    if old != new:
        changes[name] = {
            "before": old,
            "after": new,
        }

# Save installation audit
report = {
    "recorded_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "packages": packages_after,
    "dependency_changes": changes,
}

report_path = (
    AUDIT_DIR / "stage02_runtime_after_install.json"
)

report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)

print("Installed versions:")
print(json.dumps(packages_after, indent=2))

print("\nDependency changes:")
print(json.dumps(changes, indent=2))

if not changes:
    print("\nSTAGE 02 ENVIRONMENT: PASS")
else:
    print("\nSTAGE 02 ENVIRONMENT: NEEDS REVIEW")

Installed versions:
{
  "anndata": "0.13.4",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "scipy": "1.16.3",
  "h5py": "3.14.0"
}

Dependency changes:
{}

STAGE 02 ENVIRONMENT: PASS


## 3. Load and validate Stage 01 metadata

Load source tables and verify barcode, metadata and count-column ordering.


In [5]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Cell-level metadata
# --------------------------------------------------

cell_meta_raw = pd.read_csv(
    RAW_DIR / "GSE96583_batch2.total.tsne.df.tsv.gz",
    sep="\t",
    index_col=0,
    dtype={
        "ind": "string",
        "stim": "string",
        "cluster": "string",
        "cell": "string",
        "multiplets": "string",
    },
)

cell_meta_raw.index.name = "metadata_cell_id"

# --------------------------------------------------
# 2. Gene annotations
# --------------------------------------------------

genes_raw = pd.read_csv(
    RAW_DIR / "GSE96583_batch2.genes.tsv.gz",
    sep="\t",
    header=None,
    names=["gene_id", "gene_symbol"],
    dtype="string",
    keep_default_na=False,
)

# --------------------------------------------------
# 3. Original barcode lists
# --------------------------------------------------

matrix_sources = [
    ("GSM2560248", "ctrl", "GSM2560248_2.1.mtx.gz"),
    ("GSM2560249", "stim", "GSM2560249_2.2.mtx.gz"),
]

barcodes_raw = {}

for accession, condition, filename in matrix_sources:
    barcodes_raw[accession] = pd.read_csv(
        RAW_DIR / f"{accession}_barcodes.tsv.gz",
        sep="\t",
        header=None,
        names=["barcode_original"],
        dtype="string",
        keep_default_na=False,
    )["barcode_original"]

# --------------------------------------------------
# 4. Validated cell mapping from Stage 01
# --------------------------------------------------

cell_mapping = pd.read_csv(
    SOURCE_AUDIT_DIR / "cell_mapping.tsv.gz",
    sep="\t",
    dtype={
        "cell_id": "string",
        "source_sample_id": "string",
        "condition": "string",
        "matrix_column_zero_based": "int64",
        "barcode_original": "string",
        "metadata_cell_id": "string",
    },
)

counts_checks = pd.read_csv(
    SOURCE_AUDIT_DIR / "raw_count_checks.tsv",
    sep="\t",
)

print("Cell metadata:", cell_meta_raw.shape)
print("Gene metadata:", genes_raw.shape)
print("Cell mapping:", cell_mapping.shape)

print("\nBarcode counts:")
for accession, condition, _ in matrix_sources:
    print(condition, accession, len(barcodes_raw[accession]))

print("\nMETADATA LOADING: COMPLETE")

Cell metadata: (29065, 7)
Gene metadata: (35635, 2)
Cell mapping: (29065, 6)

Barcode counts:
ctrl GSM2560248 14619
stim GSM2560249 14446

METADATA LOADING: COMPLETE


In [6]:
# --------------------------------------------------
# 1. Basic dimensions and unique identifiers
# --------------------------------------------------

assert cell_meta_raw.shape == (29065, 7)
assert genes_raw.shape == (35635, 2)
assert len(cell_mapping) == 29065

assert cell_meta_raw.index.is_unique
assert genes_raw["gene_id"].is_unique

assert cell_mapping["cell_id"].is_unique
assert cell_mapping["metadata_cell_id"].is_unique

# --------------------------------------------------
# 2. Preserve original metadata order
# --------------------------------------------------

assert (
    cell_mapping["metadata_cell_id"].tolist()
    == cell_meta_raw.index.tolist()
)

# --------------------------------------------------
# 3. Verify each sample's barcode alignment
# --------------------------------------------------

alignment_results = []

for accession, condition, _ in matrix_sources:
    part = cell_mapping.loc[
        cell_mapping["source_sample_id"].eq(accession)
    ]

    original_barcodes = barcodes_raw[accession].tolist()

    expected_cell_ids = [
        accession + "__" + barcode
        for barcode in original_barcodes
    ]

    # Correct source sample and condition
    assert part["condition"].eq(condition).all()

    # Barcode order unchanged
    assert (
        part["barcode_original"].tolist()
        == original_barcodes
    )

    # Correct column positions in source matrix
    assert np.array_equal(
        part["matrix_column_zero_based"].to_numpy(),
        np.arange(len(original_barcodes)),
    )

    # Unique cell IDs reconstructed correctly
    assert part["cell_id"].tolist() == expected_cell_ids

    # Metadata IDs belong to the correct condition
    source_metadata_ids = cell_meta_raw.loc[
        cell_meta_raw["stim"].eq(condition)
    ].index.tolist()

    assert (
        part["metadata_cell_id"].tolist()
        == source_metadata_ids
    )

    n_aliases = int(
        part["barcode_original"]
        .ne(part["metadata_cell_id"])
        .sum()
    )

    alignment_results.append({
        "source_sample_id": accession,
        "condition": condition,
        "n_barcodes": len(part),
        "source_id_aliases": n_aliases,
        "alignment_pass": True,
    })

# --------------------------------------------------
# 4. Verify source metadata characteristics
# --------------------------------------------------

shared_barcodes = (
    set(barcodes_raw["GSM2560248"])
    & set(barcodes_raw["GSM2560249"])
)

assert len(shared_barcodes) == 313

assert cell_meta_raw["cell"].isna().sum() == 9

assert not cell_meta_raw[
    ["ind", "stim", "multiplets"]
].isna().any().any()

assert (
    genes_raw["gene_symbol"].duplicated().sum()
    == 2697
)

# Preserve expected conditions and sample order
assert (
    cell_mapping["source_sample_id"].tolist()
    == (
        ["GSM2560248"] * 14619
        + ["GSM2560249"] * 14446
    )
)

# --------------------------------------------------
# 5. Results
# --------------------------------------------------

alignment_df = pd.DataFrame(alignment_results)

display(alignment_df)

print("Shared barcode strings:", len(shared_barcodes))
print("Missing cell-type labels:",
      int(cell_meta_raw["cell"].isna().sum()))
print("Unique Ensembl gene IDs:",
      genes_raw["gene_id"].nunique())

print("\nSTAGE 02 METADATA ALIGNMENT: PASS")

,source_sample_id,condition,n_barcodes,source_id_aliases,alignment_pass
0,GSM2560248,ctrl,14619,0,True
1,GSM2560249,stim,14446,313,True


Shared barcode strings: 313
Missing cell-type labels: 9
Unique Ensembl gene IDs: 35635

STAGE 02 METADATA ALIGNMENT: PASS


## 4. Construct observation and gene metadata

Join cell metadata one-to-one and keep Ensembl IDs as the gene index.


In [7]:
# --------------------------------------------------
# 1. Merge mapping with original cell metadata
# --------------------------------------------------

metadata_table = cell_meta_raw.reset_index()

merged = cell_mapping.merge(
    metadata_table,
    on="metadata_cell_id",
    how="left",
    validate="one_to_one",
    sort=False,
    indicator=True,
)

# --------------------------------------------------
# 2. Verify exact alignment
# --------------------------------------------------

assert len(merged) == 29065
assert merged["_merge"].eq("both").all()

assert (
    merged["cell_id"].tolist()
    == cell_mapping["cell_id"].tolist()
)

assert (
    merged["metadata_cell_id"].tolist()
    == cell_meta_raw.index.tolist()
)

# Compare condition from mapping and source metadata
assert (
    merged["condition"].astype("string")
    == merged["stim"].astype("string")
).all()

# --------------------------------------------------
# 3. Rename fields for downstream analysis
# --------------------------------------------------

obs = (
    merged
    .drop(columns=["_merge", "stim"])
    .rename(columns={
        "ind": "donor_id",
        "cluster": "cluster_reference",
        "cell": "cell_type_reference",
        "multiplets": "demuxlet_class",
        "tsne1": "source_tsne1",
        "tsne2": "source_tsne2",
    })
    .set_index("cell_id", verify_integrity=True)
)

# --------------------------------------------------
# 4. Assign explicit data types
# --------------------------------------------------

# Preserve original identifiers as strings
obs.index = pd.Index(
    obs.index.astype(object),
    name="cell_id",
)

for col in [
    "source_sample_id",
    "barcode_original",
    "metadata_cell_id",
]:
    obs[col] = obs[col].astype(object)

# Categorical metadata
for col in [
    "donor_id",
    "cluster_reference",
    "cell_type_reference",
    "demuxlet_class",
]:
    obs[col] = pd.Categorical(obs[col])

obs["condition"] = pd.Categorical(
    obs["condition"],
    categories=["ctrl", "stim"],
)

# --------------------------------------------------
# 5. Final checks
# --------------------------------------------------

assert obs.shape[0] == 29065
assert obs.index.is_unique

assert (
    obs.index.tolist()
    == cell_mapping["cell_id"].tolist()
)

assert obs["donor_id"].notna().all()
assert obs["condition"].notna().all()

assert (
    obs["cell_type_reference"].isna().sum()
    == 9
)

print("obs shape:", obs.shape)
print("Unique cell IDs:", obs.index.is_unique)
print("Missing cell types:",
      int(obs["cell_type_reference"].isna().sum()))

display(obs.head(3))

print("\nOBS CONSTRUCTION: PASS")

obs shape: (29065, 11)
Unique cell IDs: True
Missing cell types: 9


,source_sample_id,condition,matrix_column_zero_based,barcode_original,metadata_cell_id,source_tsne1,source_tsne2,donor_id,cluster_reference,cell_type_reference,demuxlet_class
cell_id,,,,,,,,,,,
GSM2560248__AAACATACAATGCC-1,GSM2560248,ctrl,0,AAACATACAATGCC-1,AAACATACAATGCC-1,-4.277833,-19.294709,107,5,CD4 T cells,doublet
GSM2560248__AAACATACATTTCC-1,GSM2560248,ctrl,1,AAACATACATTTCC-1,AAACATACATTTCC-1,-27.640373,14.966629,1016,9,CD14+ Monocytes,singlet
GSM2560248__AAACATACCAGAAA-1,GSM2560248,ctrl,2,AAACATACCAGAAA-1,AAACATACCAGAAA-1,-27.493646,28.924885,1256,9,CD14+ Monocytes,singlet



OBS CONSTRUCTION: PASS


In [8]:
# --------------------------------------------------
# 1. Construct gene metadata
# --------------------------------------------------

var = (
    genes_raw
    .set_index("gene_id", verify_integrity=True)
    [["gene_symbol"]]
    .copy()
)

var.index = pd.Index(
    var.index.astype(object),
    name="gene_id",
)

var["gene_symbol"] = var["gene_symbol"].astype(object)

# --------------------------------------------------
# 2. Validate gene alignment
# --------------------------------------------------

assert var.shape == (35635, 1)
assert var.index.is_unique

assert (
    var.index.tolist()
    == genes_raw["gene_id"].tolist()
)

assert (
    var["gene_symbol"].tolist()
    == genes_raw["gene_symbol"].tolist()
)

assert (
    var["gene_symbol"].duplicated().sum()
    == 2697
)

# --------------------------------------------------
# 3. Verify donor-condition distribution
# --------------------------------------------------

donor_condition = pd.crosstab(
    obs["donor_id"],
    obs["condition"],
).reindex(
    columns=["ctrl", "stim"],
    fill_value=0,
)

# Independent comparison with source metadata
source_distribution = pd.crosstab(
    cell_meta_raw["ind"],
    cell_meta_raw["stim"],
).reindex(
    index=donor_condition.index,
    columns=["ctrl", "stim"],
    fill_value=0,
)

assert np.array_equal(
    donor_condition.to_numpy(),
    source_distribution.to_numpy(),
)

assert donor_condition.shape == (8, 2)
assert donor_condition.to_numpy().sum() == 29065

# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

print("var shape:", var.shape)
print("Unique Ensembl IDs:", var.index.is_unique)
print("Duplicated gene-symbol rows:",
      int(var["gene_symbol"].duplicated().sum()))

print("\nGene metadata preview:")
display(var.head(3))

print("\nDonor-condition distribution:")
display(donor_condition)

print("\nOBS/VAR AND DONOR ALIGNMENT: PASS")

var shape: (35635, 1)
Unique Ensembl IDs: True
Duplicated gene-symbol rows: 2697

Gene metadata preview:


,gene_symbol
gene_id,
ENSG00000243485,MIR1302-10
ENSG00000237613,FAM138A
ENSG00000186092,OR4F5



Donor-condition distribution:


condition,ctrl,stim
donor_id,,
101,1027,1363
1015,3138,2703
1016,2180,1998
1039,552,797
107,645,641
1244,2250,1755
1256,2451,2275
1488,2376,2914



OBS/VAR AND DONOR ALIGNMENT: PASS


## 5. Restore and validate raw UMI count matrices

Verify sparse integer counts against the Stage 01 audit.


In [9]:
import gzip
import numpy as np
import pandas as pd

from scipy.io import mmread
from scipy.sparse import issparse

# --------------------------------------------------
# 1. Restore source counts
# --------------------------------------------------

counts_by_sample = {}
count_records = []

for accession, condition, filename in matrix_sources:

    print(f"Reading {accession} ({condition})", flush=True)

    # Read original Matrix Market matrix
    with gzip.open(RAW_DIR / filename, "rb") as stream:
        source_counts = mmread(stream, spmatrix=True)

    # --------------------------------------------------
    # 2. Verify shape and sparse representation
    # --------------------------------------------------

    expected_shape = (
        len(genes_raw),
        len(barcodes_raw[accession]),
    )

    assert issparse(source_counts), accession
    assert source_counts.shape == expected_shape, accession

    # --------------------------------------------------
    # 3. Validate original count values
    # --------------------------------------------------

    values = source_counts.data

    finite = bool(np.isfinite(values).all())
    nonnegative = bool((values >= 0).all())

    integer_valued = bool(
        (values == np.floor(values)).all()
    )

    assert finite, f"Non-finite counts: {accession}"
    assert nonnegative, f"Negative counts: {accession}"
    assert integer_valued, f"Fractional counts: {accession}"

    if values.size:
        assert values.max() < 2**63, (
            f"Counts outside int64 range: {accession}"
        )

    # --------------------------------------------------
    # 4. Convert to CSR int64
    # --------------------------------------------------

    counts = source_counts.astype(np.int64).tocsr()

    # CSR conversion can combine duplicate coordinates
    duplicate_coordinates = int(
        source_counts.nnz - counts.nnz
    )

    assert duplicate_coordinates == 0, (
        f"Duplicate matrix coordinates: {accession}"
    )

    # Count explicitly stored zeros before removing them
    explicit_zeros = int((counts.data == 0).sum())

    counts.eliminate_zeros()

    # --------------------------------------------------
    # 5. Preserve validated matrix
    # --------------------------------------------------

    assert counts.dtype == np.int64
    assert counts.format == "csr"
    assert counts.shape == expected_shape

    counts_by_sample[accession] = counts

    count_records.append({
        "source_sample_id": accession,
        "condition": condition,
        "n_genes": counts.shape[0],
        "n_barcodes": counts.shape[1],
        "source_dtype": str(source_counts.dtype),
        "validated_dtype": str(counts.dtype),
        "finite": finite,
        "nonnegative": nonnegative,
        "integer_valued": integer_valued,
        "duplicate_coordinates": duplicate_coordinates,
        "explicit_zero_entries": explicit_zeros,
        "nnz": int(counts.nnz),
        "total_umi_all_source_barcodes": int(counts.sum()),
    })

    print(
        f"  shape={counts.shape}, "
        f"nnz={counts.nnz}, "
        f"UMI={int(counts.sum())}"
    )

    del source_counts, values

# --------------------------------------------------
# 6. Results
# --------------------------------------------------

stage02_counts_checks = pd.DataFrame(count_records)

assert len(counts_by_sample) == 2

display(stage02_counts_checks)

print("\nRAW COUNTS RELOADED: PASS")

Reading GSM2560248 (ctrl)
  shape=(35635, 14619), nnz=8732747, UMI=23540929
Reading GSM2560249 (stim)
  shape=(35635, 14446), nnz=8838098, UMI=25059996


,source_sample_id,condition,n_genes,n_barcodes,source_dtype,validated_dtype,finite,nonnegative,integer_valued,duplicate_coordinates,explicit_zero_entries,nnz,total_umi_all_source_barcodes
0,GSM2560248,ctrl,35635,14619,float64,int64,True,True,True,0,0,8732747,23540929
1,GSM2560249,stim,35635,14446,float64,int64,True,True,True,0,0,8838098,25059996



RAW COUNTS RELOADED: PASS


In [10]:
# --------------------------------------------------
# 1. Compare against saved Stage 01 count audit
# --------------------------------------------------

assert len(stage02_counts_checks) == len(counts_checks)

assert (
    stage02_counts_checks.columns.tolist()
    == counts_checks.columns.tolist()
)

expected = counts_checks.set_index(
    "source_sample_id",
    verify_integrity=True,
)

observed = stage02_counts_checks.set_index(
    "source_sample_id",
    verify_integrity=True,
)

assert observed.index.tolist() == expected.index.tolist()

for accession in observed.index:
    for column in observed.columns:

        actual = observed.loc[accession, column]
        reference = expected.loc[accession, column]

        assert actual == reference, (
            f"Audit mismatch: {accession}, {column}, "
            f"observed={actual}, expected={reference}"
        )

# --------------------------------------------------
# 2. Check matrix-to-metadata alignment
# --------------------------------------------------

start = 0

for accession, condition, _ in matrix_sources:

    matrix = counts_by_sample[accession]
    n_barcodes = matrix.shape[1]

    part = obs.iloc[start:start + n_barcodes]

    assert len(part) == n_barcodes

    assert (
        part["source_sample_id"] == accession
    ).all()

    assert (
        part["condition"] == condition
    ).all()

    assert (
        part["barcode_original"].tolist()
        == barcodes_raw[accession].tolist()
    )

    assert np.array_equal(
        part["matrix_column_zero_based"].to_numpy(),
        np.arange(n_barcodes),
    )

    assert matrix.shape[0] == len(var)

    start += n_barcodes

assert start == len(obs)

# --------------------------------------------------
# 3. Verify global count statistics
# --------------------------------------------------

total_nnz = sum(
    matrix.nnz
    for matrix in counts_by_sample.values()
)

total_umi = sum(
    int(matrix.sum())
    for matrix in counts_by_sample.values()
)

assert total_nnz == 17570845
assert total_umi == 48600925

# --------------------------------------------------
# 4. Save Stage 02 audit
# --------------------------------------------------

audit_path = (
    AUDIT_DIR / "stage02_count_reload_checks.tsv"
)

stage02_counts_checks.to_csv(
    audit_path,
    sep="\t",
    index=False,
)

print("Validated matrices:", len(counts_by_sample))
print("Total nonzero entries:", total_nnz)
print("Total raw UMI:", total_umi)
print("Metadata alignment: PASS")
print("Stage 01 count audit comparison: PASS")
print("Audit saved:", audit_path)

print("\nSTAGE 02 RAW COUNTS VALIDATION: PASS")

Validated matrices: 2
Total nonzero entries: 17570845
Total raw UMI: 48600925
Metadata alignment: PASS
Stage 01 count audit comparison: PASS
Audit saved: /kaggle/working/kang_ifnb/results/audit/stage02_count_reload_checks.tsv

STAGE 02 RAW COUNTS VALIDATION: PASS


## 6. Construct the unfiltered AnnData object

Stack control then stimulated cells. Verify the raw count layer and donor labels.


In [11]:
import anndata as ad
import numpy as np
from scipy import sparse

# --------------------------------------------------
# 1. Transpose source matrices
# --------------------------------------------------

count_blocks = []

for accession, condition, _ in matrix_sources:

    source = counts_by_sample[accession]

    assert source.format == "csr"
    assert source.dtype == np.int64

    # gene x barcode -> barcode x gene
    block = source.T.tocsr()

    assert block.shape == (
        len(barcodes_raw[accession]),
        len(var),
    )

    assert block.dtype == np.int64

    count_blocks.append(block)

# --------------------------------------------------
# 2. Stack CTRL followed by STIM
# --------------------------------------------------

X_all = sparse.vstack(
    count_blocks,
    format="csr",
)

assert X_all.shape == (29065, 35635)
assert X_all.dtype == np.int64

# --------------------------------------------------
# 3. Construct AnnData
# --------------------------------------------------

adata = ad.AnnData(
    X=X_all,
    obs=obs.copy(),
    var=var.copy(),
)

# Independent copy of unnormalized UMI counts
adata.layers["counts"] = adata.X.copy()

# --------------------------------------------------
# 4. Record data provenance
# --------------------------------------------------

adata.uns["provenance"] = {
    "geo_series": "GSE96583",
    "source_batch": "batch2",
    "source_sample_order": [
        "GSM2560248",
        "GSM2560249",
    ],
    "count_type": "unnormalized UMI counts",
    "count_dtype": "int64",
    "source_orientation": "gene_by_barcode",
    "anndata_orientation": "barcode_by_gene",
    "qc_filtered": False,
    "primary_analysis": "donor-aware CD14+ monocytes",
}

# --------------------------------------------------
# 5. Initial verification
# --------------------------------------------------

assert adata.shape == (29065, 35635)
assert adata.X.format == "csr"
assert adata.X.dtype == np.int64

assert (
    adata.obs_names.tolist()
    == obs.index.tolist()
)

assert (
    adata.var_names.tolist()
    == var.index.tolist()
)

assert "counts" in adata.layers
assert adata.layers["counts"].dtype == np.int64
assert adata.raw is None

print(adata)

print("\nX shape:", adata.X.shape)
print("X format:", adata.X.format)
print("X dtype:", adata.X.dtype)

print(
    "Counts layer:",
    adata.layers["counts"].shape
)

print("\nANNADATA CONSTRUCTION: PASS")

AnnData object with n_obs × n_vars = 29065 × 35635
    obs: 'source_sample_id', 'condition', 'matrix_column_zero_based', 'barcode_original', 'metadata_cell_id', 'source_tsne1', 'source_tsne2', 'donor_id', 'cluster_reference', 'cell_type_reference', 'demuxlet_class'
    var: 'gene_symbol'
    uns: 'provenance'
    layers: None (.X), 'counts'

X shape: (29065, 35635)
X format: csr
X dtype: int64
Counts layer: (29065, 35635)

ANNADATA CONSTRUCTION: PASS


In [12]:
# --------------------------------------------------
# 1. Compare each AnnData block with source counts
# --------------------------------------------------

start = 0

for accession, condition, _ in matrix_sources:

    source = counts_by_sample[accession]

    n_cells = source.shape[1]
    end = start + n_cells

    actual = adata.layers["counts"][start:end, :]
    expected = source.T.tocsr()

    assert actual.shape == expected.shape
    assert actual.dtype == expected.dtype
    assert actual.nnz == expected.nnz

    # Exact numerical comparison without densifying
    assert (actual != expected).nnz == 0

    # Verify source sample and barcode alignment
    part = adata.obs.iloc[start:end]

    assert part["source_sample_id"].eq(accession).all()
    assert part["condition"].eq(condition).all()

    assert (
        part["barcode_original"].tolist()
        == barcodes_raw[accession].tolist()
    )

    print(
        f"{accession}: COUNTS AND ALIGNMENT PASS"
    )

    start = end

assert start == adata.n_obs

# --------------------------------------------------
# 2. Verify total UMI counts
# --------------------------------------------------

assert adata.X.nnz == 17570845
assert int(adata.X.sum()) == 48600925

assert (
    adata.layers["counts"].nnz
    == 17570845
)

assert (
    int(adata.layers["counts"].sum())
    == 48600925
)

assert (
    adata.X != adata.layers["counts"]
).nnz == 0

# --------------------------------------------------
# 3. Verify donor pairing among source singlets
# --------------------------------------------------

singlets = adata.obs.loc[
    adata.obs["demuxlet_class"].eq("singlet")
]

singlet_pairs = pd.crosstab(
    singlets["donor_id"],
    singlets["condition"],
).reindex(
    columns=["ctrl", "stim"],
    fill_value=0,
)

assert len(singlets) == 24679
assert singlet_pairs.shape == (8, 2)

# All 8 source donor labels occur in both conditions
assert (singlet_pairs > 0).all().all()

# --------------------------------------------------
# 4. Results
# --------------------------------------------------

print("\nSource singlets by donor and condition:")
display(singlet_pairs)

print("\nAnnData shape:", adata.shape)
print("Total nonzero entries:", adata.X.nnz)
print("Total UMI:", int(adata.X.sum()))
print("Source singlet donors:", len(singlet_pairs))

print("\nANNADATA INTEGRITY: PASS")

GSM2560248: COUNTS AND ALIGNMENT PASS
GSM2560249: COUNTS AND ALIGNMENT PASS

Source singlets by donor and condition:


condition,ctrl,stim
donor_id,,
101,858,1166
1015,2738,2352
1016,1723,1635
1039,461,647
107,517,525
1244,1879,1464
1256,2108,2026
1488,2031,2549



AnnData shape: (29065, 35635)
Total nonzero entries: 17570845
Total UMI: 48600925
Source singlet donors: 8

ANNADATA INTEGRITY: PASS


## 7. H5AD serialization and round-trip verification

Reload the candidate H5AD and check exact counts, identifiers and metadata before promotion.


In [13]:
from pathlib import Path
import anndata as ad
import numpy as np

# --------------------------------------------------
# 1. Define output paths
# --------------------------------------------------

candidate_path = (
    PROCESSED_DIR / "ifnb_batch2_candidate.h5ad"
)

final_path = (
    PROCESSED_DIR / "ifnb_batch2_raw_counts.h5ad"
)

# --------------------------------------------------
# 2. Verify object before writing
# --------------------------------------------------

assert adata.shape == (29065, 35635)
assert adata.X.format == "csr"
assert adata.X.dtype == np.int64

assert "counts" in adata.layers

assert adata.layers["counts"].dtype == np.int64

assert adata.X.nnz == 17570845
assert int(adata.X.sum()) == 48600925

assert adata.raw is None

# --------------------------------------------------
# 3. Write candidate file
# --------------------------------------------------

adata.write_h5ad(
    candidate_path,
    compression="lzf",
    convert_strings_to_categoricals=False,
)

# --------------------------------------------------
# 4. Verify file exists
# --------------------------------------------------

assert candidate_path.is_file()
assert candidate_path.stat().st_size > 0

print("Candidate file:", candidate_path)
print("File size (MB):",
      round(candidate_path.stat().st_size / 1e6, 2))

print("\nH5AD CANDIDATE WRITE: PASS")

Candidate file: /kaggle/working/kang_ifnb/data/processed/ifnb_batch2_candidate.h5ad
File size (MB): 155.04

H5AD CANDIDATE WRITE: PASS


In [14]:
import json
import hashlib
import pandas as pd
import numpy as np
from scipy import sparse
from importlib.metadata import version

# --------------------------------------------------
# 1. Read candidate H5AD
# --------------------------------------------------

loaded = ad.read_h5ad(candidate_path)

assert loaded.shape == adata.shape

print("H5AD loaded:", loaded.shape)

# --------------------------------------------------
# 2. Verify cell and gene identities
# --------------------------------------------------

assert loaded.obs_names.tolist() == adata.obs_names.tolist()
assert loaded.var_names.tolist() == adata.var_names.tolist()

assert loaded.obs_names.is_unique
assert loaded.var_names.is_unique

print("Cell and gene identifiers: PASS")

# --------------------------------------------------
# 3. Verify all metadata values
# --------------------------------------------------

assert loaded.obs.columns.tolist() == adata.obs.columns.tolist()
assert loaded.var.columns.tolist() == adata.var.columns.tolist()

# Compare values, allowing equivalent pandas storage dtypes
# while requiring exact metadata values and missingness
pd.testing.assert_frame_equal(
    loaded.obs,
    adata.obs,
    check_dtype=False,
    check_categorical=False,
    check_index_type=False,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    loaded.var,
    adata.var,
    check_dtype=False,
    check_categorical=False,
    check_index_type=False,
    check_exact=True,
)

assert int(loaded.obs["cell_type_reference"].isna().sum()) == 9

print("Cell and gene metadata: PASS")

# --------------------------------------------------
# 4. Verify sparse matrices
# --------------------------------------------------

def assert_same_counts(actual, expected):
    assert sparse.issparse(actual)
    assert sparse.issparse(expected)

    assert actual.format == expected.format == "csr"
    assert actual.shape == expected.shape
    assert actual.dtype == expected.dtype == np.int64
    assert actual.nnz == expected.nnz

    # Exact element-by-element comparison, without densifying
    assert (actual != expected).nnz == 0


assert_same_counts(loaded.X, adata.X)

assert "counts" in loaded.layers

assert_same_counts(
    loaded.layers["counts"],
    adata.layers["counts"],
)

assert_same_counts(
    loaded.X,
    loaded.layers["counts"],
)

assert loaded.raw is None

print("Raw count matrices: PASS")

# --------------------------------------------------
# 5. Verify each sample against original source counts
# --------------------------------------------------

start = 0

for accession, condition, _ in matrix_sources:

    original = counts_by_sample[accession]
    n_cells = original.shape[1]

    end = start + n_cells

    restored_block = loaded.layers["counts"][start:end, :]

    assert_same_counts(
        restored_block,
        original.T.tocsr(),
    )

    restored_obs = loaded.obs.iloc[start:end]

    assert restored_obs["source_sample_id"].eq(accession).all()
    assert restored_obs["condition"].eq(condition).all()

    assert (
        restored_obs["barcode_original"].tolist()
        == barcodes_raw[accession].tolist()
    )

    print(f"{accession}: SOURCE MATCH PASS")

    start = end

assert start == loaded.n_obs

# --------------------------------------------------
# 6. Verify donor pairing after reload
# --------------------------------------------------

def get_singlet_pairs(obj):
    selected = obj.obs.loc[
        obj.obs["demuxlet_class"].eq("singlet")
    ]

    return pd.crosstab(
        selected["donor_id"].astype(str),
        selected["condition"].astype(str),
    ).reindex(
        columns=["ctrl", "stim"],
        fill_value=0,
    ).sort_index()


original_pairs = get_singlet_pairs(adata)
restored_pairs = get_singlet_pairs(loaded)

pd.testing.assert_frame_equal(
    original_pairs,
    restored_pairs,
)

assert restored_pairs.shape == (8, 2)
assert int(restored_pairs.to_numpy().sum()) == 24679
assert (restored_pairs > 0).all().all()

print("Donor-condition singlet pairing: PASS")

# --------------------------------------------------
# 7. Verify provenance
# --------------------------------------------------

provenance = loaded.uns["provenance"]

assert provenance["geo_series"] == "GSE96583"
assert provenance["source_batch"] == "batch2"

assert list(provenance["source_sample_order"]) == [
    "GSM2560248",
    "GSM2560249",
]

assert not bool(provenance["qc_filtered"])

print("Provenance: PASS")

# --------------------------------------------------
# 8. Verify global count statistics
# --------------------------------------------------

assert loaded.X.nnz == 17570845
assert int(loaded.X.sum()) == 48600925

assert loaded.layers["counts"].nnz == 17570845
assert int(loaded.layers["counts"].sum()) == 48600925

print("Global count statistics: PASS")

# --------------------------------------------------
# 9. Promote verified file to final checkpoint
# --------------------------------------------------

candidate_path.replace(final_path)

assert final_path.is_file()

# Calculate checksum of the final H5AD file
with final_path.open("rb") as file:
    file_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

# --------------------------------------------------
# 10. Save round-trip audit
# --------------------------------------------------

report = {
    "anndata_version": version("anndata"),
    "output_file": str(final_path),
    "file_size_bytes": final_path.stat().st_size,
    "sha256": file_sha256,
    "n_obs": int(loaded.n_obs),
    "n_vars": int(loaded.n_vars),
    "counts_dtype": str(loaded.layers["counts"].dtype),
    "counts_format": loaded.layers["counts"].format,
    "counts_nnz": int(loaded.layers["counts"].nnz),
    "total_umi": int(loaded.layers["counts"].sum()),
    "cell_gene_alignment": "PASS",
    "metadata_roundtrip": "PASS",
    "source_count_comparison": "PASS",
    "donor_pairing": "PASS",
    "provenance": "PASS",
    "roundtrip": "PASS",
}

report_path = (
    AUDIT_DIR / "stage02_anndata_roundtrip.json"
)

report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)

print("\nVerified file:", final_path)
print("SHA-256:", file_sha256)

print("\nH5AD ROUND TRIP: PASS")

H5AD loaded: (29065, 35635)
Cell and gene identifiers: PASS
Cell and gene metadata: PASS
Raw count matrices: PASS
GSM2560248: SOURCE MATCH PASS
GSM2560249: SOURCE MATCH PASS
Donor-condition singlet pairing: PASS
Provenance: PASS
Global count statistics: PASS

Verified file: /kaggle/working/kang_ifnb/data/processed/ifnb_batch2_raw_counts.h5ad
SHA-256: 7fe3231e8c53354c233a3b16943fa6196e6b37ef74a3c1471c1087c9ccfc6153

H5AD ROUND TRIP: PASS


## 8. Stage 02 checkpoint verification

Verify the saved H5AD digest and required audit files.


In [15]:
import json
import hashlib

required_outputs = [
    PROCESSED_DIR / "ifnb_batch2_raw_counts.h5ad",
    AUDIT_DIR / "stage02_runtime_before_install.json",
    AUDIT_DIR / "stage02_runtime_after_install.json",
    AUDIT_DIR / "stage02_count_reload_checks.tsv",
    AUDIT_DIR / "stage02_anndata_roundtrip.json",
]

for path in required_outputs:
    assert path.is_file(), f"Missing: {path}"
    assert path.stat().st_size > 0

    print(
        path.relative_to(PROJECT),
        f"({path.stat().st_size:,} bytes)"
    )

# Cross-check final H5AD checksum with saved audit
manifest = json.loads(
    (AUDIT_DIR / "stage02_anndata_roundtrip.json")
    .read_text(encoding="utf-8")
)

h5ad_path = PROCESSED_DIR / "ifnb_batch2_raw_counts.h5ad"

with h5ad_path.open("rb") as stream:
    actual_sha256 = hashlib.file_digest(
        stream, "sha256"
    ).hexdigest()

assert actual_sha256 == manifest["sha256"]
assert manifest["roundtrip"] == "PASS"

print("\nH5AD SHA-256:", actual_sha256)
print("Checkpoint files:", len(required_outputs))
print("\nSTAGE 02 CHECKPOINT: PASS")

data/processed/ifnb_batch2_raw_counts.h5ad (155,042,332 bytes)
results/audit/stage02_runtime_before_install.json (214 bytes)
results/audit/stage02_runtime_after_install.json (223 bytes)
results/audit/stage02_count_reload_checks.tsv (343 bytes)
results/audit/stage02_anndata_roundtrip.json (552 bytes)

H5AD SHA-256: 7fe3231e8c53354c233a3b16943fa6196e6b37ef74a3c1471c1087c9ccfc6153
Checkpoint files: 5

STAGE 02 CHECKPOINT: PASS
